# What Drives the Price of a Car?
## CRISP-DM Framework: Practical Application 2

**Business Understanding:**  
A used car dealership wants to understand what factors most influence the sale price of a used car. By identifying key price drivers, the dealership can make smarter inventory acquisition decisions, optimize pricing strategy, and better serve customers.

We will apply the CRISP-DM framework to analyze a dataset of 426K used vehicles from Kaggle and provide actionable recommendations.

## 1. Business Understanding

**Goal:** Identify the key features that make a car more or less expensive.

**Success Criteria:** Build a regression model that predicts used car prices accurately and identifies the most impactful features. Provide clear, actionable recommendations to the client.

**Evaluation Metric:** RMSE on log-transformed price. Chosen because price is right-skewed and RMSE on the log scale penalizes proportional errors, which is appropriate for price prediction across a wide value range.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded successfully.')

## 2. Data Understanding

The dataset was sourced from Kaggle and contains information on 426K used cars scraped from Craigslist. Features include price, year, manufacturer, model, condition, odometer, fuel type, drive type, and more.

> **Note:** Download vehicles.csv from [Kaggle](https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data) and place it in the data/ folder.

In [ ]:
# Load dataset
df = pd.read_csv('data/vehicles.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['price'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Price Distribution (Raw)', fontsize=14)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

np.log1p(df['price']).hist(bins=50, ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Price Distribution (Log Scale)', fontsize=14)
axes[1].set_xlabel('log(Price)')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable: Price', fontsize=16)
plt.tight_layout()
plt.savefig('images/price_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features vs median price
cat_cols = ['condition', 'fuel', 'title_status', 'transmission', 'drive', 'type']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, col in zip(axes.flatten(), cat_cols):
    order = df.groupby(col)['price'].median().sort_values().index
    df.groupby(col)['price'].median().loc[order].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Median Price by {col.title()}', fontsize=12)
    ax.set_xlabel('Median Price ($)')
    ax.set_ylabel(col.title())

plt.suptitle('Median Price by Categorical Features', fontsize=16)
plt.tight_layout()
plt.savefig('images/categorical_vs_price.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Continuous features vs price
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['year'], df['price'], alpha=0.05, s=1, color='steelblue')
axes[0].set_title('Year vs Price', fontsize=14)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Price ($)')

axes[1].scatter(df['odometer'], df['price'], alpha=0.05, s=1, color='darkorange')
axes[1].set_title('Odometer vs Price', fontsize=14)
axes[1].set_xlabel('Odometer (miles)')
axes[1].set_ylabel('Price ($)')

plt.suptitle('Continuous Features vs Price', fontsize=16)
plt.tight_layout()
plt.savefig('images/continuous_vs_price.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Data Preparation

Steps:
- Drop irrelevant or high-missing-value columns
- Filter out extreme outliers in price, year, and odometer
- Drop rows with missing values in key features
- Log-transform the target variable (price)

In [ ]:
# Drop irrelevant columns
drop_cols = ['id', 'url', 'region_url', 'VIN', 'image_url',
             'description', 'county', 'lat', 'long', 'posting_date', 'size']
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Filter price outliers
df_clean = df_clean[(df_clean['price'] > 500) & (df_clean['price'] < 150000)]

# Filter year range
df_clean = df_clean[(df_clean['year'] >= 1990) & (df_clean['year'] <= 2023)]

# Filter odometer
df_clean = df_clean[df_clean['odometer'] < 400000]

# Drop rows missing key features
key_cols = ['year', 'odometer', 'condition', 'fuel', 'title_status',
            'transmission', 'drive', 'type', 'paint_color', 'cylinders']
df_clean = df_clean.dropna(subset=key_cols)

# Log-transform price
df_clean['log_price'] = np.log1p(df_clean['price'])

print(f'Cleaned dataset shape: {df_clean.shape}')
df_clean.head()

In [ ]:
# Correlation heatmap
numeric_df = df_clean[['year', 'odometer', 'log_price']]

plt.figure(figsize=(7, 5))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix (Numeric Features)', fontsize=14)
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Modeling

We train three regression models:
1. **Linear Regression** - baseline model
2. **Ridge Regression** - L2 regularization, tuned with GridSearchCV
3. **Lasso Regression** - L1 regularization, tuned with GridSearchCV

**Evaluation Metric:** RMSE on log(price). Lower RMSE = better model.

In [ ]:
# Define features and target
features = ['year', 'odometer', 'condition', 'fuel', 'title_status',
            'transmission', 'drive', 'type', 'paint_color', 'cylinders']
target = 'log_price'

df_model = df_clean[features + [target]].dropna()
X = df_model[features]
y = df_model[target]

num_features = ['year', 'odometer']
cat_features = ['condition', 'fuel', 'title_status', 'transmission',
                'drive', 'type', 'paint_color', 'cylinders']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Model 1: Linear Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_cv = cross_val_score(lr_pipeline, X_train, y_train, cv=5,
                        scoring='neg_root_mean_squared_error')

print(f'Linear Regression - Test RMSE: {lr_rmse:.4f}')
print(f'Linear Regression - CV RMSE (mean +/- std): {-lr_cv.mean():.4f} +/- {lr_cv.std():.4f}')

In [ ]:
# Model 2: Ridge Regression with GridSearchCV
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

ridge_params = {'model__alpha': [0.1, 1.0, 10.0, 100.0]}
ridge_grid = GridSearchCV(ridge_pipeline, ridge_params, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train, y_train)

ridge_pred = ridge_grid.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print(f'Ridge Best Alpha: {ridge_grid.best_params_}')
print(f'Ridge Regression - Test RMSE: {ridge_rmse:.4f}')

In [ ]:
# Model 3: Lasso Regression with GridSearchCV
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=10000))
])

lasso_params = {'model__alpha': [0.001, 0.01, 0.1, 1.0]}
lasso_grid = GridSearchCV(lasso_pipeline, lasso_params, cv=5,
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train, y_train)

lasso_pred = lasso_grid.predict(X_test)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

print(f'Lasso Best Alpha: {lasso_grid.best_params_}')
print(f'Lasso Regression - Test RMSE: {lasso_rmse:.4f}')

In [ ]:
# Model comparison
model_results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Lasso Regression'],
    'Test RMSE (log price)': [lr_rmse, ridge_rmse, lasso_rmse]
})
model_results = model_results.sort_values('Test RMSE (log price)')
print(model_results.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(model_results['Model'], model_results['Test RMSE (log price)'],
        color=['steelblue', 'darkorange', 'green'])
plt.title('Model Comparison: Test RMSE (log price)', fontsize=14)
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.tight_layout()
plt.savefig('images/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Evaluation

**Evaluation Metric: RMSE on log(price)**

We use log-transformed price as the target because raw price is heavily right-skewed. RMSE on the log scale measures proportional prediction error, which is more meaningful when prices range from hundreds to over $100,000.

Ridge and Lasso regularization help prevent overfitting by penalizing large coefficients. GridSearchCV selects the best regularization strength via 5-fold cross-validation.

In [ ]:
# Actual vs Predicted (best model - Ridge)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, ridge_pred, alpha=0.1, s=2, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual log(Price)', fontsize=12)
plt.ylabel('Predicted log(Price)', fontsize=12)
plt.title('Ridge Regression: Actual vs Predicted log(Price)', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig('images/actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - ridge_pred

plt.figure(figsize=(8, 5))
plt.scatter(ridge_pred, residuals, alpha=0.1, s=2, color='darkorange')
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted log(Price)', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Ridge Regression: Residuals vs Predicted', fontsize=14)
plt.tight_layout()
plt.savefig('images/residuals.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Findings and Recommendations

### Key Findings

Based on our analysis of 426K used car listings using the CRISP-DM framework:

1. **Vehicle Age (Year)** is the strongest positive predictor of price. Newer vehicles command significantly higher prices.
2. **Odometer Reading** has a strong negative effect. Every additional 10,000 miles reduces value meaningfully. Low-mileage vehicles (under 80,000 miles) hold premium value.
3. **Drive Type:** 4WD and AWD vehicles are priced significantly higher than FWD/RWD equivalents.
4. **Fuel Type:** Diesel and electric vehicles are priced at a premium compared to gasoline vehicles.
5. **Vehicle Type:** Pickup trucks and offroad vehicles consistently command the highest median prices; hatchbacks and sedans the lowest.
6. **Condition:** Vehicles in like-new and excellent condition command a significant premium over fair or salvage condition vehicles.
7. **Title Status:** Clean titles are strongly preferred by buyers. Salvage or rebuilt titles significantly reduce a vehicle's market value.

### Recommendations for Used Car Dealers

- **Prioritize newer inventory:** Focus acquisition on vehicles under 5 years old.
- **Mileage matters:** Target vehicles with under 80,000 miles for the strongest pricing leverage.
- **Stock 4WD trucks and SUVs:** These command the highest prices and attract price-insensitive buyers.
- **Avoid salvage titles:** Salvage and rebuilt title vehicles are significantly harder to price competitively.
- **Condition reconditioning:** Investing in bringing a vehicle from good to excellent condition can meaningfully increase sale price.
- **Diesel and electric vehicles:** Consider stocking more diesel trucks and EVs as they command a market premium.

### Next Steps

- Incorporate manufacturer and model as features using target encoding to reduce cardinality.
- Experiment with gradient boosting models (XGBoost, LightGBM) for improved accuracy.
- Perform geographic analysis to assess regional pricing effects.
- Build an interactive pricing tool for the dealership using the best-performing model.